# GPU Runtime Required

This notebook is intended to run on Google Colab with a GPU runtime enabled.

Before running all cells, go to **Runtime -> Change runtime type -> Hardware accelerator -> GPU**.
Running this notebook on CPU can be impractically slow and may not reproduce the expected representation-learning output within normal Colab limits.

TensorFlow is intentionally not included in the local `pyproject.toml` dependencies because this representation-learning stage is expected to run in a Colab GPU environment.

# MvDEC Representation Learning

This notebook trains the MvDEC representation-learning stage and saves the fused representation used by the downstream FP-Max and clustering pipeline.

In [ ]:
# Install Colab runtime dependencies.
!pip install -q tensorflow openpyxl pandas scikit-learn matplotlib

In [ ]:
import pickle
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.cluster import KMeans
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import silhouette_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Concatenate, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UnicodeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 42
N_ITERATIONS = 20
N_CLUSTERS = 5
BATCH_SIZE = 128
EPOCHS = 100
VALIDATION_SPLIT = 0.1
EARLY_STOPPING_PATIENCE = 10
LEARNING_RATE = 1e-4

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "No GPU runtime detected. In Colab, select Runtime -> Change runtime "
        "type -> GPU."
    )

print("GPU devices:", gpus)

In [ ]:
def find_project_dir(start: Path | None = None) -> Path:
    """Find the project root that contains pyproject.toml and data/."""

    if start is None:
        start = Path.cwd()

    candidates = [start, *start.parents]
    for base in [start, *start.parents]:
        candidates.extend(
            [
                base / "mvdec_fpmax",
                base / "mvdec_fpmax" / "mvdec_fpmax",
            ]
        )

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from the "
        "mvdec_fpmax project directory or update PROJECT_DIR manually."
    )


PROJECT_DIR = find_project_dir()
PREPROCESSED_DATA_DIR = PROJECT_DIR / "data" / "preprocessed_data"
INPUT_DATA_PATH = PREPROCESSED_DATA_DIR / "tiki_preprocessed.csv"
OUTPUT_REPRESENTATION_PATH = PREPROCESSED_DATA_DIR / "fused_representation.pkl"

print("Project directory:", PROJECT_DIR)
print("Input data:", INPUT_DATA_PATH)
print("Output representation:", OUTPUT_REPRESENTATION_PATH)

In [ ]:
if not INPUT_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing preprocessed data file: {INPUT_DATA_PATH}")

df = pd.read_csv(INPUT_DATA_PATH)
X = df.values.astype(np.float32)
input_dim = X.shape[1]

print("Data shape:", X.shape)
print("Input dimension:", input_dim)
display(df.head(2))

In [ ]:
def build_view1_models(input_dim: int = 7, latent_dim: int = 4) -> tuple[Model, Model]:
    """Build the MLP autoencoder view and its embedding model."""

    inputs = Input(shape=(input_dim,), name="view1_input")

    encoded = Dense(250, activation="relu")(inputs)
    encoded = Dense(250, activation="relu")(encoded)
    encoded = Dense(1000, activation="relu")(encoded)
    bottleneck = Dense(latent_dim, activation="linear", name="bottleneck_view1")(
        encoded
    )

    decoded = Dense(1000, activation="relu")(bottleneck)
    decoded = Dense(250, activation="relu")(decoded)
    decoded = Dense(250, activation="relu")(decoded)
    reconstruction = Dense(
        input_dim,
        activation="linear",
        name="reconstruction_view1",
    )(decoded)

    autoencoder = Model(inputs=inputs, outputs=reconstruction, name="View1_Autoencoder")
    embedding_output = Concatenate(name="view1_embedding")([bottleneck, reconstruction])
    embedding_model = Model(
        inputs=inputs,
        outputs=embedding_output,
        name="View1_EmbeddingModel",
    )
    return autoencoder, embedding_model


def build_view2_models(input_dim: int = 7) -> tuple[Model, Model]:
    """Build the U-Net-style autoencoder view and its embedding model."""

    inputs = Input(shape=(input_dim,), name="view2_input")

    x1 = Dense(32, activation="relu")(inputs)
    x2 = Dense(32, activation="relu")(x1)
    x3 = Dense(64, activation="relu")(x2)
    x4 = Dense(64, activation="relu")(x3)
    x5 = Dense(128, activation="relu")(x4)
    x6 = Dense(128, activation="relu")(x5)
    x7 = Dense(256, activation="relu")(x6)
    x8 = Dense(256, activation="relu")(x7)
    x9 = Dense(512, activation="relu")(x8)
    x10 = Dense(256, activation="relu")(x9)
    x11 = Dense(128, activation="relu")(x10)

    decoded = Concatenate()([x8, x11])
    decoded = Dense(256, activation="relu")(decoded)
    decoded = Dense(128, activation="relu")(decoded)
    decoded = Dense(64, activation="relu")(decoded)
    decoded = Concatenate()([decoded, x6])
    decoded = Dense(128, activation="relu")(decoded)
    decoded = Dense(64, activation="relu")(decoded)
    decoded = Dense(32, activation="relu")(decoded)
    decoded = Concatenate()([decoded, x3])
    decoded = Dense(64, activation="relu")(decoded)
    decoded = Dense(32, activation="relu")(decoded)
    decoded = Dense(16, activation="relu")(decoded)
    decoded = Concatenate()([decoded, x1])
    decoded = Dense(32, activation="relu")(decoded)

    embedding = Dense(11, activation=None, name="view2_embedding")(decoded)
    reconstruction = Dense(
        input_dim,
        activation="linear",
        name="reconstruction_view2",
    )(embedding)

    autoencoder = Model(inputs=inputs, outputs=reconstruction, name="View2_Autoencoder")
    embedding_model = Model(
        inputs=inputs, outputs=embedding, name="View2_EmbeddingModel"
    )
    return autoencoder, embedding_model

In [ ]:
scores_kmeanspp: list[float] = []
scores_random: list[float] = []
best_score = -1.0
best_labels = None
best_init = None
best_iteration = -1
best_fused = None

for iteration in range(N_ITERATIONS):
    print(f"\nIteration {iteration + 1}/{N_ITERATIONS}")

    auto_v1, embed_v1 = build_view1_models(input_dim=X.shape[1])
    auto_v2, embed_v2 = build_view2_models(input_dim=X.shape[1])
    auto_v1.compile(optimizer=Adam(LEARNING_RATE), loss="mse")
    auto_v2.compile(optimizer=Adam(LEARNING_RATE), loss="mse")

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )
    auto_v1.fit(
        X,
        X,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_split=VALIDATION_SPLIT,
        callbacks=[early_stop],
        verbose=0,
    )
    auto_v2.fit(
        X,
        X,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_split=VALIDATION_SPLIT,
        callbacks=[early_stop],
        verbose=0,
    )

    h1 = embed_v1.predict(X, verbose=0)
    h2 = embed_v2.predict(X, verbose=0)
    h_fused = (h1 + h2) / 2

    for init_method in ["k-means++", "random"]:
        kmeans = KMeans(
            n_clusters=N_CLUSTERS,
            init=init_method,
            n_init=10,
            random_state=SEED,
        )
        labels = kmeans.fit_predict(h_fused)
        score = silhouette_score(h_fused, labels)
        print(f"{init_method:10s} | Silhouette: {score:.4f}")

        if init_method == "k-means++":
            scores_kmeanspp.append(float(score))
        else:
            scores_random.append(float(score))

        if score > best_score:
            best_score = float(score)
            best_labels = labels
            best_fused = h_fused
            best_init = init_method
            best_iteration = iteration + 1

print("\nBest score:", best_score)
print("Best init:", best_init)
print("Best iteration:", best_iteration)

In [ ]:
if best_fused is None or best_labels is None:
    raise RuntimeError("Training did not produce a fused representation.")

best_result = {
    "h_fused": best_fused,
    "labels": best_labels,
    "init": best_init,
    "score": best_score,
    "iteration": best_iteration,
}

OUTPUT_REPRESENTATION_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_REPRESENTATION_PATH.open("wb") as file:
    pickle.dump(best_result, file)

print(f"Saved fused representation to: {OUTPUT_REPRESENTATION_PATH}")

In [ ]:
with OUTPUT_REPRESENTATION_PATH.open("rb") as file:
    saved_result = pickle.load(file)

print("Saved keys:", sorted(saved_result.keys()))
print("h_fused shape:", saved_result["h_fused"].shape)
print("labels shape:", saved_result["labels"].shape)
print("init:", saved_result["init"])
print("score:", saved_result["score"])
print("iteration:", saved_result["iteration"])

In [ ]:
iterations = range(1, len(scores_kmeanspp) + 1)
ymin = min(min(scores_kmeanspp), min(scores_random))
ymax = max(max(scores_kmeanspp), max(scores_random))
padding = 0.02

plt.figure(figsize=(10, 5))
plt.plot(iterations, scores_kmeanspp, marker="o", label="k-means++")
plt.plot(iterations, scores_random, marker="s", label="random")
plt.title("Silhouette Score over Representation-Learning Iterations")
plt.xlabel("Iteration")
plt.ylabel("Silhouette Score")
plt.xticks(list(iterations))
plt.ylim(ymin - padding, ymax + padding)
plt.grid(False)
plt.legend()
plt.tight_layout()
plt.show()